In [1]:
import numpy as np
import plotly.graph_objects as go

def calculate_min_sample_size(target_cvr, expected_cvr):
    """
    Calculates the minimum sample size needed to prove that
    expected_cvr > target_cvr with 95% confidence (1-sided).
    """
    
    # 1. Validation Logic
    if expected_cvr <= target_cvr:
        print(f"❌ ERROR: Your Expected CVR ({expected_cvr:.1%}) must be HIGHER than your Target ({target_cvr:.1%}) to plan a success test.")
        return

    # 2. The Formula (Frequentist Approximation for 95% Confidence)
    # N = (1.65^2 * p * (1-p)) / (p - target)^2
    # 1.65^2 is approx 2.72
    z_score_squared = 1.65**2
    numerator = z_score_squared * expected_cvr * (1 - expected_cvr)
    denominator = (expected_cvr - target_cvr)**2
    
    required_n = int(np.ceil(numerator / denominator))
    
    # 3. Print Results
    print(f"--- SAMPLE SIZE PLANNER ---")
    print(f"Target Viability: {target_cvr:.1%}")
    print(f"Expected Reality: {expected_cvr:.1%}")
    print(f"-" * 30)
    print(f"You need roughly {required_n:,} visitors.")
    print(f"-" * 30)

    # 4. Generate Data for the 'Planning Curve'
    # We plot a range of possible 'Expected Conversion Rates' to see how N changes
    # Start slightly above the target (e.g., target + 0.1%) up to target + 5%
    x_values = np.linspace(target_cvr + 0.002, target_cvr + 0.10, 500)
    
    # Calculate N for all these hypothetical rates
    y_values = (z_score_squared * x_values * (1 - x_values)) / ((x_values - target_cvr)**2)

    # 5. Create Interactive Plotly Graph
    fig = go.Figure()
    
    # Define Theme Colors
    primary_blue = '#1f77b4'

    # A. The Main Curve (Sample Size vs CVR)
    fig.add_trace(go.Scatter(
        x=x_values, y=y_values,
        mode='lines',
        name='Required Traffic',
        line=dict(color=primary_blue, width=3)
    ))

    # B. Your Specific Point (The Dot)
    fig.add_trace(go.Scatter(
        x=[expected_cvr], y=[required_n],
        mode='markers',
        name='Your Plan',
        marker=dict(color='red', size=12, symbol='diamond')
    ))

    # C. Layout Formatting
    fig.update_layout(
        title=dict(
            text=f"<b>Minimum Sample Size Planner</b><br>Target Viability: {target_cvr:.1%}",
            font=dict(size=20)
        ),
        xaxis_title="If your True Conversion Rate is...",
        yaxis_title="Visitors Needed (N)",
        xaxis=dict(tickformat=".1%", showgrid=True, gridcolor='#eee'),
        yaxis=dict(showgrid=True, gridcolor='#eee'), # range=[0, max(required_n*2, 1000)]
        template="plotly_white",
        showlegend=True,
        plot_bgcolor='rgba(0,0,0,0)'
    )

    # Add annotation for the user's point
    fig.add_annotation(
        x=expected_cvr, y=required_n,
        text=f"<b>{required_n} Visitors</b><br>needed at {expected_cvr:.1%}",
        ax=40, ay=-40,
        font=dict(size=12, color="red"),
        arrowcolor="red"
    )

    fig.show()

# ==========================================
# ENTER YOUR DATA HERE
# ==========================================
# Example: You need 1%, you hope for 2%
calculate_min_sample_size(target_cvr=0.01, expected_cvr=0.02)

--- SAMPLE SIZE PLANNER ---
Target Viability: 1.0%
Expected Reality: 2.0%
------------------------------
You need roughly 534 visitors.
------------------------------
